# Challenge 1: Temporal Mismatch

| Product | Temporal Resolution | Area |
| :------- | :------: | :------: |
| MOD13Q1 | 16 days | 16-day composite | 
| MOD11A2 | 8 days | 8-day composite |

MOD11A2 has 2 values for every 1 MOD13Q1 value.

We can't directly align the NDVI, EVI and LST values unless they are from the same time period. We need to aggregate MOD11A2 to 16-day. Combine each pair of 8-day LST composites into one 16-day composite to match MOD13Q1.

Example:

lst_1 = np.array([...])  # Shape: (1200, 1200) LST from days 1–8

lst_2 = np.array([...])  # Shape: (1200, 1200) LST from days 9–16

lst_16day = np.nanmean([lst_1, lst_2], axis=0)  # Shape: (1200, 1200)

**Import Packages**

In [753]:

from datetime import datetime, timedelta
from pyhdf.SD import SD, SDC
from pyproj import Proj
import re
import matplotlib.pyplot as plt
import numpy as np 
import os
import pandas as pd

**Extract the date from a MOD13Q1 filename and for a given list of MOD11A2 files, find the 2 MOD11A2 files that overlap with the MOD13Q1 16-day window**

In [757]:
def parse_day_of_year(filepath):
    """
    Extracts the date from a MODIS filename like 'MOD13Q1.A2020241.h08v04...'
    """
    filename = os.path.basename(filepath)
    parts = filename.split('.')
    date_str = parts[1]  # example: A2020241
    year = int(date_str[1:5])
    doy = int(date_str[5:])
    return datetime(year, 1, 1) + timedelta(days=doy - 1)


def get_matching_mod11a2_files(mod13q1_filename, mod11a2_file_list):
    """
    Given a MOD13Q1 filename and a list of MOD11A2 filenames,
    return the MOD11A2 files that temporally overlap with the MOD13Q1 16-day window.
    """
    mod13_start = parse_day_of_year(mod13q1_filename)
    mod13_end = mod13_start + timedelta(days=15)

    matching_files = []
    for mod11_file in mod11a2_file_list:
        mod11_start = parse_day_of_year(mod11_file)
        mod11_end = mod11_start + timedelta(days=7)  # 8-day window

        # Temporal overlap check
        if mod11_start <= mod13_end and mod11_end >= mod13_start:
            matching_files.append(mod11_file)

    return matching_files

**Process MOD11A2 - LST (1km)**

In [758]:
def process_single_mod11a2_file(file_path):
    """
    Load and process MOD11A2 LST data (1km), apply scale factor,
    convert to Celsius, and return daytime and nighttime LST as separate arrays.
    
    Returns:
        Tuple of two NumPy arrays: (lst_day_celsius, lst_night_celsius), each shape (1200, 1200)
    """
    
    # Load the MOD11A2 tile file
    hdf = SD(file_path, SDC.READ)
    
    # Load daytime and nighttime LST datasets
    lst_day = hdf.select('LST_Day_1km')[:]
    lst_night = hdf.select('LST_Night_1km')[:]
    
    #print(f"MOD11A2 LST Day original shape: {lst_day.shape}")
    #print(f"MOD11A2 LST Night original shape: {lst_night.shape}")

    # Replace fill values (0) with NaN
    lst_day = np.where(lst_day == 0, np.nan, lst_day)
    lst_night = np.where(lst_night == 0, np.nan, lst_night)

    # Apply scale factor: 0.02, then convert Kelvin to Celsius
    lst_day_celsius = (lst_day * 0.02) - 273.15
    lst_night_celsius = (lst_night * 0.02) - 273.15

    hdf.end()
    return lst_day_celsius, lst_night_celsius

**compute a mask-weighted average**

In [759]:
def safe_average(arrays):
    """Compute pixel-wise average of arrays, ignoring NaNs."""
    arrays = np.stack(arrays)
    valid_mask = ~np.isnan(arrays)
    sum_valid = np.nansum(arrays, axis=0)
    count_valid = np.sum(valid_mask, axis=0)
    with np.errstate(invalid='ignore', divide='ignore'):
        avg = np.divide(sum_valid, count_valid)
    avg[count_valid == 0] = np.nan
    return avg

**Process 2 MOD11A2 Files (8-day composite each) into a combined 16-day composite**

In [1]:
def process_combined_mod11a2(mod11a2_files,file_path, target_tile, min_valid_pixels=100000):
    """
    Given a list of matched MOD11A2 files and a target tile (h, v),
    return the average LST Day and Night arrays only for that tile.

    Args:
        mod11a2_files (list): List of MOD11A2 filenames
        file_path: path to MOD11A2 files
        target_tile (tuple): (h, v) tile ID, e.g. (9, 4)
        min_valid_pixels (int): Minimum number of valid pixels required

    Returns:
        Tuple: (lst_day_combined, lst_night_combined)
    """
    h_target, v_target = target_tile
    day_list = []
    night_list = []

    for f in mod11a2_files:
        tile_id = f.split('.')[2]  # example: 'h09v04'
        h = int(tile_id[1:3])
        v = int(tile_id[4:6])

        if (h, v) != target_tile:
            continue  # skip files not in the same tile

        mod11a2_filename = os.path.join(file_path, f)
        try:
            day, night = process_single_mod11a2_file(mod11a2_filename)
        except Exception as e:
            print(f"Error reading {f}: {e}")
            continue

        valid_day = np.sum(~np.isnan(day))
        valid_night = np.sum(~np.isnan(night))
        print(f"{f} - LST Day valid: {valid_day}, LST Night valid: {valid_night}")

        if valid_day < min_valid_pixels or valid_night < min_valid_pixels:
            print(f"Skipping {f} due to insufficient valid pixels.")
            continue

        day_list.append(day)
        night_list.append(night)

    if not day_list or not night_list:
        raise ValueError(f"No valid MOD11A2 files for tile h{h_target:02d}v{v_target:02d}")

    # Combine via nanmean
    lst_day_combined = safe_average(day_list)
    lst_night_combined = safe_average(night_list)

    return lst_day_combined, lst_night_combined